# 01 — Data inspection

Load SCP1219 and verify structure, metadata columns, condition labels, donor counts before running the pipeline. Use this notebook once after download to populate `config.yaml` with the correct column names.

In [ ]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.utils import load_config
from src.io import load_atlas

cfg = load_config()
adata = load_atlas(cfg)
print(adata)

In [ ]:
# Inspect metadata columns
print('Columns:', adata.obs.columns.tolist())
adata.obs.head()

In [ ]:
# Condition & cell-type value counts
for c in adata.obs.select_dtypes(include=['object','category']).columns:
    vc = adata.obs[c].value_counts()
    if 1 < len(vc) < 50:
        print(c); print(vc); print()

In [ ]:
# Donor counts by condition
donor = cfg['batch_correction']['batch_key']
cond = cfg['conditions']['condition_column']
if donor in adata.obs and cond in adata.obs:
    print(adata.obs.groupby(cond)[donor].nunique())
    print(adata.obs[donor].value_counts().head(20))

In [ ]:
# Gene presence check for core markers
markers = ['SFTPC','SFTPA1','AGER','PDPN','HOPX','KRT8','CLDN4','ISG15','TNF','BAX','CASP3']
present = [g for g in markers if g in adata.var_names]
missing = [g for g in markers if g not in adata.var_names]
print(f'Present: {present}')
print(f'Missing: {missing}')